<a href="https://colab.research.google.com/github/adityab-tech/PRISMx/blob/main/notebooks/03_Market_Conditioned_Gating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import os
import sys
import yaml
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 371 (delta 66), reused 49 (delta 49), pack-reused 269 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 26.61 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
!pip -q install transformers peft

In [13]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [19]:
from peft import LoraConfig, get_peft_model, TaskType
from model import Kronos ,KronosTokenizer

In [28]:
PROJECT_ROOT = "/content/drive/MyDrive/PRISM"
TOKENIZER_PATH = f"{PROJECT_ROOT}/models/kronos/tokenizer/best_model"
KRONOS_PATH = f"{PROJECT_ROOT}/models/kronos/basemodel/best_model"
DATASET_PATH = f"{PROJECT_ROOT}/datasets"
RESULTS_PATH = f"{PROJECT_ROOT}/results/del3"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [32]:
#Load Tokenizer
tokenizer = KronosTokenizer.from_pretrained("/content/drive/MyDrive/PRISM/models/kronos/tokenizer/best_model",local_files_only=True)
tokenizer.eval()
print("Tokenizer Loaded")

#Load Fine Tuned Kronos
model = Kronos.from_pretrained( "/content/drive/MyDrive/PRISM/models/kronos/basemodel/best_model",local_files_only=True)
model.eval()
print("Kronos Loaded")

Loading weights from local directory
Tokenizer Loaded
Loading weights from local directory
Kronos Loaded


In [33]:
#Freeze tokenizer
for param in tokenizer.parameters():
    param.requires_grad = False
#Freeze Kronos backbone
for param in model.parameters():
    param.requires_grad = False

In [43]:
lora_config = LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",
    target_modules=["q_proj","k_proj","v_proj","out_proj",])

In [44]:
!pip install -U torchao

In [45]:
model = get_peft_model(model,lora_config)

In [46]:
print(type(tokenizer))
print(type(model))

<class 'model.kronos.KronosTokenizer'>
<class 'peft.peft_model.PeftModel'>


In [47]:
model.print_trainable_parameters()

trainable params: 1,384,448 || all params: 103,695,040 || trainable%: 1.3351


In [48]:
lora_layers = []
for name, _ in model.named_parameters():
    if "lora_" in name:
        lora_layers.append(name)

print("Total LoRA tensors :", len(lora_layers))
print(lora_layers[:10])

Total LoRA tensors : 104
['base_model.model.base_model.model.base_model.model.transformer.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.out_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.0.self_attn.out_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.transformer.1.self_attn.q_proj.lora_A.default.weight', 'base_m

In [49]:
market_df = pd.read_csv("/content/drive/MyDrive/PRISM/data/processed/NIFTY50_processed.csv")
market_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PRISM/data/processed/NIFTY50_processed.csv'